# Week 1: Single LIF Neuron + Reproducibility Convention

This notebook runs top-to-bottom on a fresh Colab runtime. It:
1. Installs `snnkit` (editable install from the repo).
2. Establishes the reproducibility convention used throughout this repo (fixed seed, recorded package versions).
3. Simulates a single LIF neuron and validates it against the analytical firing-rate formula.

**To run on Colab:** File -> Save a copy in Drive, then Runtime -> Run all.

In [ ]:
# If running on Colab, clone/install the repo. If running locally with the
# package already installed (e.g. via `pip install -e .`), this cell is a no-op.
import importlib.util
if importlib.util.find_spec("snnkit") is None:
    !git clone https://github.com/YOUR_USERNAME/snnkit.git
    %cd snnkit
    !pip install -e . --quiet

In [ ]:
from snnkit.reproducibility import set_seed, get_package_versions
import json

key = set_seed(0)  # fixed seed convention used throughout this repo
print("Environment / package versions (record this alongside any reported result):")
print(json.dumps(get_package_versions(), indent=2))

## Simulate a single LIF neuron

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt

from snnkit.core.neuron import LIFParams, simulate_lif, analytical_firing_rate

params = LIFParams(tau=20e-3, v_th=1.0, v_reset=0.0, r=1.0, dt=1e-3)
duration_s = 1.0
n_steps = int(duration_s / params.dt)

I_input = 1.5  # constant input current, above threshold so the neuron fires
i_trace = jnp.full((n_steps,), I_input)

v_trace, spike_trace = simulate_lif(i_trace, params)

t = jnp.arange(n_steps) * params.dt
fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
axes[0].plot(t, v_trace)
axes[0].axhline(params.v_th, color="red", linestyle="--", label="threshold")
axes[0].set_ylabel("v")
axes[0].legend()
spike_times = t[spike_trace > 0]
axes[1].eventplot(spike_times, lineoffsets=0, linelengths=0.8)
axes[1].set_ylabel("spikes")
axes[1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

## Validate against the analytical firing-rate formula

In [ ]:
sim_rate = float(spike_trace.sum()) / duration_s
analytic_rate = float(analytical_firing_rate(I_input, params))
rel_error = abs(sim_rate - analytic_rate) / analytic_rate

print(f"Simulated firing rate:  {sim_rate:.2f} Hz")
print(f"Analytical firing rate: {analytic_rate:.2f} Hz")
print(f"Relative error:         {rel_error*100:.2f}%")

assert rel_error < 0.05, "simulated rate should match the analytical formula within 5%"
print("\nMatches analytical formula within tolerance.")

## Firing rate vs. input current

A quick sweep confirming the simulated f-I curve tracks the analytical one across a range of inputs, not just the one value checked above.

In [ ]:
import numpy as np

I_values = np.linspace(1.01, 3.0, 15)
sim_rates, analytic_rates = [], []
for I_val in I_values:
    i_trace = jnp.full((n_steps,), float(I_val))
    _, spikes = simulate_lif(i_trace, params)
    sim_rates.append(float(spikes.sum()) / duration_s)
    analytic_rates.append(float(analytical_firing_rate(I_val, params)))

plt.figure(figsize=(6, 4))
plt.plot(I_values, analytic_rates, label="analytical", linewidth=2)
plt.scatter(I_values, sim_rates, label="simulated", color="orange", zorder=3)
plt.xlabel("input current I")
plt.ylabel("firing rate (Hz)")
plt.legend()
plt.title("LIF f-I curve: simulated vs. analytical")
plt.tight_layout()
plt.show()